# Preprocessing

Load the train, validation, and test splits, then fit preprocessing on the train split only and reuse the learned parameters for validation and test.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

data_dir = Path("../data/processed")

train_df = pd.read_csv(data_dir / "train.csv")
validation_df = pd.read_csv(data_dir / "validation.csv")
test_df = pd.read_csv(data_dir / "test.csv")

train_df.shape, validation_df.shape, test_df.shape

((315000, 20), (67500, 20), (67500, 20))

## Train-Only Preprocessing Plan

- Fit numeric medians and the `price_usd` p99 cap on `train_df` only.
- Impute numeric missing values with train medians.
- Clip `price_usd` at the train p99 cap.
- Use one-hot encoding for low-cardinality categoricals.
- Use frequency encoding for high-cardinality categoricals.
- Apply the same fitted parameters to validation and test.


In [2]:
target = "clicked"
onehot_cols = ["platform", "device_os", "page_type", "day_of_week", "gender", "category"]
freq_cols = ["city", "product_name", "brand"]
numeric_cols = ["slot_position", "hour_of_day", "age", "price_usd", "discount_pct", "avg_rating", "review_count"]


def fit_preprocessor(train_frame: pd.DataFrame):
    features = train_frame.drop(columns=[target])
    numeric_medians = features[numeric_cols].median(numeric_only=True)
    categorical_fill = "missing"
    price_cap = features["price_usd"].quantile(0.99)

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoder.fit(features[onehot_cols].fillna(categorical_fill).astype(str))

    freq_maps = {}
    for col in freq_cols:
        freq_maps[col] = features[col].fillna(categorical_fill).astype(str).value_counts(normalize=True).to_dict()

    return {
        "numeric_medians": numeric_medians,
        "categorical_fill": categorical_fill,
        "price_cap": price_cap,
        "encoder": encoder,
        "freq_maps": freq_maps,
    }


def transform_frame(frame: pd.DataFrame, fitted: dict) -> pd.DataFrame:
    features = frame.drop(columns=[target]).copy()

    numeric_part = features[numeric_cols].copy().fillna(fitted["numeric_medians"])
    numeric_part["price_usd"] = numeric_part["price_usd"].clip(upper=fitted["price_cap"])

    onehot_part = features[onehot_cols].copy().fillna(fitted["categorical_fill"]).astype(str)
    encoded = fitted["encoder"].transform(onehot_part)
    encoded_cols = fitted["encoder"].get_feature_names_out(onehot_cols)
    encoded_df = pd.DataFrame(encoded, columns=encoded_cols, index=frame.index)

    freq_encoded = pd.DataFrame(index=frame.index)
    for col in freq_cols:
        freq_encoded[f"{col}_freq"] = features[col].fillna(fitted["categorical_fill"]).astype(str).map(fitted["freq_maps"][col]).fillna(0.0)

    cleaned = pd.concat(
        [frame[[target]].reset_index(drop=True), numeric_part.reset_index(drop=True), freq_encoded.reset_index(drop=True), encoded_df.reset_index(drop=True)],
        axis=1,
    )
    return cleaned


fitted = fit_preprocessor(train_df)
clean_train = transform_frame(train_df, fitted)
clean_validation = transform_frame(validation_df, fitted)
clean_test = transform_frame(test_df, fitted)

clean_train.shape, clean_validation.shape, clean_test.shape

((315000, 38), (67500, 38), (67500, 38))

In [3]:
clean_train.to_csv(data_dir / "train_cleaned.csv", index=False)
clean_validation.to_csv(data_dir / "validation_cleaned.csv", index=False)
clean_test.to_csv(data_dir / "test_cleaned.csv", index=False)

print("Saved:")
print(data_dir / "train_cleaned.csv")
print(data_dir / "validation_cleaned.csv")
print(data_dir / "test_cleaned.csv")


Saved:
../data/processed/train_cleaned.csv
../data/processed/validation_cleaned.csv
../data/processed/test_cleaned.csv
